In [2]:
from LanguageDatasets import LanguageDataset
from dotenv import load_dotenv
import os

### 1. Traducir un Q&A a los idiomas

### 2. Traducción como tarea de evaluación

La traducción gallego ↔ español, asturiano ↔ español y aranés ↔ español permite medir comprensión y generación. Para el aranés puede ser útil incluir también comparaciones con francés, dado su parentesco occitano. 

### 3. Corrección gramatical mediante introducción de errores

Para evaluar la capacidad de corrección, se parte de textos correctos en asturiano o aranés (por ejemplo, de Wikipedia). Un LLM grande introduce errores controlados de ortografía, morfología o sintaxis. El modelo evaluado debe corregirlos. La comparación con el texto original permite medir la calidad de la corrección.

- ```"llama-3.1-8b-instant"``` Mucho más rápido, pruebas
- ```"openai/gpt-oss-120b"``` Para dataset final
- ```"qwen/qwen3-32b"``` Probar

In [ ]:
from generateDataset import generateDatasetOrtograficoAnotado
load_dotenv("secrets.env")
ast = LanguageDataset("asturiano",True)
evalDataset = generateDatasetOrtograficoAnotado(ast, os.getenv("GROQ_API_KEY"), save=False, model="llama-3.1-8b-instant")

Descargando tatoeba para asturiano:
Completado con éxito
Progreso:  76.8% (344/448) | step: 3 s, remaining time: 5 min 12 sssError en intento 1: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kdjbyd5aeftbs2nrxfpm9e93` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99981, Requested 122. Please try again in 1m28.992s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}. Reintentando en 0.15 segundos...
Error en intento 2: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kdjbyd5aeftbs2nrxfpm9e93` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99981, Requested 122. Please try again in 1m28.992s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_lim

In [4]:
evalDataset.shape

(344, 3)

In [1]:
evalDataset.head()

NameError: name 'evalDataset' is not defined

In [ ]:
evalDataset.n_errors.value_counts()

#### Evaluación

### 4. Medición de castellanización o interferencia lingüística

Para detectar si el modelo mezcla castellano con asturiano o aranés

In [ ]:
def mergeFreelingDictionary(folder_path, output_file):
    """
    Combina todos los archivos .src del diccionario de FreeLing
    en un único archivo con una palabra por línea.
    Optimizado para velocidad.
    """
    dictionary_path = os.path.join(folder_path, "dictionary")
    lexicon = set()

    for filename in os.listdir(dictionary_path):

        full_path = os.path.join(dictionary_path, filename)

        with open(full_path, "r", encoding="utf8", errors="ignore") as f:
            for line in f:
                if not line or line.startswith("#"):
                    continue
                word = line.split(" ", 1)[0].lower() 
                lexicon.add(word)

    with open(output_file, "w", encoding="utf8") as out:
        out.write("\n".join(sorted(lexicon)))

    print(f"Lexicón generado: {output_file} ({len(lexicon)} palabras)")


#### 4.1. Índice tipo‑token (TTR)

Se calcula como número de palabras únicas dividido entre el total de palabras. Un TTR bajo puede indicar uso excesivo de vocabulario castellano básico.

In [ ]:
ttr(text)

#### 4.2. Entropía léxica

Se calcula la distribución de frecuencias de los tokens y su entropía. Una entropía baja sugiere un vocabulario poco variado y potencial castellanización.

#### 4.3. Frecuencia relativa de formas propias

Se construye un lexicón asturiano o aranés a partir de corpus públicos. Se compara la proporción de tokens generados por el modelo que pertenecen al lexicón propio frente a un lexicón castellano o francés (en el caso del aranés). Esto permite medir interferencia.

#### 4.4. N‑gram overlap con corpus de referencia

Se toma un corpus real en asturiano o aranés (por ejemplo, Wikipedia). Se extraen sus n‑gramas y se comparan con los n‑gramas generados por el modelo. El modelo no recibe nada en esta fase; simplemente se analizan sus salidas. Un solapamiento bajo indica que el modelo no reproduce patrones característicos de la lengua.
